# 第8章：记忆（Memory）

## 概念

**记忆** = 让 LLM 能够记住之前的对话内容

```
用户: "我叫小明"
AI: "你好，小明！"
用户: "我叫什么名字？"
AI: "你叫小明。"  ← 记住了之前的对话
```

## 记忆类型

| 类型 | 说明 | 特点 |
|------|------|------|
| **手动管理** | 使用列表保存对话历史 | 简单、灵活 |
| **向量存储记忆** | 将对话存入向量数据库 | 可检索，适合长期记忆 |

## 代码演示

使用 LangChain 实现手动管理对话历史

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

In [3]:
# 加载环境变量
load_dotenv()

# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## 方式1：手动管理对话历史

使用列表保存对话历史，手动传递给 LLM

In [4]:
# 手动管理对话历史
chat_history = []

# 定义提示词
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手。请根据对话历史回答问题。"),
    ("placeholder", "{chat_history}"),
    ("user", "{question}")
])

# 创建链
chain = prompt | llm | StrOutputParser()

# 定义对话函数
def chat(question):
    # 调用 LLM
    response = chain.invoke({
        "chat_history": chat_history,
        "question": question
    })
    
    # 保存对话历史
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    
    return response

# 测试对话
print("## 方式1：手动管理对话历史 ##")
print("\n用户: 我叫小明")
print("AI:", chat("我叫小明"))

print("\n用户: 我叫什么名字？")
print("AI:", chat("我叫什么名字？"))

print("\n用户: 今天天气怎么样？")
print("AI:", chat("今天天气怎么样？"))

## 方式1：手动管理对话历史 ##

用户: 我叫小明
AI: 你好，小明！很高兴认识你！😊  
有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都在这里哦～

用户: 我叫什么名字？
AI: 你叫**小明**！😊  
有什么需要我帮忙的吗？

用户: 今天天气怎么样？
AI: 抱歉，我无法获取实时的天气信息，因为我没有联网功能，也不知道你所在的位置。😊

建议你可以：
- 查看手机上的**天气应用**
- 搜索"**你所在城市 + 天气**"
- 问问身边的朋友或家人

你在哪里呢？如果告诉我城市，我可以聊聊那个地方的一般气候特点～


## 方式2：使用字典保存对话历史

使用字典保存对话历史，更结构化

In [5]:
# 使用字典保存对话历史
conversation_log = []

# 定义提示词
dict_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手。请根据对话历史回答问题。"),
    ("placeholder", "{chat_history}"),
    ("user", "{question}")
])

# 创建链
dict_chain = dict_prompt | llm | StrOutputParser()

# 定义对话函数
def chat_with_dict(question):
    # 转换为消息格式
    messages = []
    for entry in conversation_log:
        messages.append(HumanMessage(content=entry["user"]))
        messages.append(AIMessage(content=entry["ai"]))
    
    # 调用 LLM
    response = dict_chain.invoke({
        "chat_history": messages,
        "question": question
    })
    
    # 保存到对话日志
    conversation_log.append({"user": question, "ai": response})
    
    return response

# 测试对话
print("## 方式2：使用字典保存对话历史 ##")
print("\n用户: 我叫小红")
print("AI:", chat_with_dict("我叫小红"))

print("\n用户: 我叫什么名字？")
print("AI:", chat_with_dict("我叫什么名字？"))

print("\n用户: 我们之前聊了什么？")
print("AI:", chat_with_dict("我们之前聊了什么？"))

print("\n对话日志：")
for entry in conversation_log:
    print(f"用户: {entry['user']}")
    print(f"AI: {entry['ai']}")

## 方式2：使用字典保存对话历史 ##

用户: 我叫小红
AI: 你好，小红！很高兴认识你！😊  
有什么我可以帮你的吗？无论是学习、生活问题，还是想聊聊兴趣爱好，我都在这里哦～

用户: 我叫什么名字？
AI: 你叫**小红**呀！这是你刚才告诉我的～ 😊

用户: 我们之前聊了什么？
AI: 我们之前的对话很简单：

1. 你告诉我你叫**小红**
2. 你问我你叫什么名字
3. 我回答你叫小红

就这些啦～ 我们刚刚才开始聊天呢！😄 有什么想聊的吗？

对话日志：
用户: 我叫小红
AI: 你好，小红！很高兴认识你！😊  
有什么我可以帮你的吗？无论是学习、生活问题，还是想聊聊兴趣爱好，我都在这里哦～
用户: 我叫什么名字？
AI: 你叫**小红**呀！这是你刚才告诉我的～ 😊
用户: 我们之前聊了什么？
AI: 我们之前的对话很简单：

1. 你告诉我你叫**小红**
2. 你问我你叫什么名字
3. 我回答你叫小红

就这些啦～ 我们刚刚才开始聊天呢！😄 有什么想聊的吗？


## 方式3：限制对话历史长度

只保留最近的几轮对话，节省内存

In [6]:
# 限制对话历史长度
max_history = 3  # 只保留最近3轮对话
limited_history = []

# 定义提示词
limited_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手。请根据对话历史回答问题。"),
    ("placeholder", "{chat_history}"),
    ("user", "{question}")
])

# 创建链
limited_chain = limited_prompt | llm | StrOutputParser()

# 定义对话函数
def chat_with_limited_history(question):
    # 调用 LLM
    response = limited_chain.invoke({
        "chat_history": limited_history,
        "question": question
    })
    
    # 保存对话历史
    limited_history.append(HumanMessage(content=question))
    limited_history.append(AIMessage(content=response))
    
    # 限制历史长度
    if len(limited_history) > max_history * 2:
        limited_history.pop(0)
        limited_history.pop(0)
    
    return response

# 测试对话
print("## 方式3：限制对话历史长度 ##")
print("\n用户: 我叫小华")
print("AI:", chat_with_limited_history("我叫小华"))

print("\n用户: 我喜欢编程")
print("AI:", chat_with_limited_history("我喜欢编程"))

print("\n用户: 我在学习Python")
print("AI:", chat_with_limited_history("我在学习Python"))

print("\n用户: 我叫什么名字？")
print("AI:", chat_with_limited_history("我叫什么名字？"))

print("\n用户: 我喜欢什么？")
print("AI:", chat_with_limited_history("我喜欢什么？"))

print("\n当前历史长度：", len(limited_history))

## 方式3：限制对话历史长度 ##

用户: 我叫小华
AI: 你好，小华！很高兴认识你！😊 有什么我可以帮你的吗？无论是问题、聊天，还是需要建议，我都在这里哦～

用户: 我喜欢编程
AI: 太棒了，小华！编程是一项非常酷且实用的技能，它能帮你把想法变成现实，解决各种问题。😊

你目前主要在用什么编程语言呢？或者你对哪个方向特别感兴趣？比如：

*   **Web开发**（前端：JavaScript/TypeScript；后端：Python/Java/Go等）
*   **移动应用开发**（iOS/Swift， Android/Kotlin， 跨平台/Flutter/React Native）
*   **数据科学/人工智能**（Python， R）
*   **游戏开发**（C#， C++）
*   **系统/嵌入式开发**（C， C++， Rust）

无论你是刚开始学习，还是已经是个经验丰富的开发者，我都很乐意和你聊聊编程相关的话题，比如：

*   **学习路径**：推荐一些好的学习资源或项目。
*   **解决问题**：遇到代码bug或者设计难题时一起探讨。
*   **技术讨论**：聊聊最新的技术趋势、框架或工具。
*   **项目灵感**：如果你想找点项目练手，我们可以一起头脑风暴。

**编程的乐趣在于创造和解决问题，而最大的挑战往往也伴随着最大的成就感。** 请随时告诉我你正在做什么，或者你对什么感到好奇吧！

用户: 我在学习Python
AI: 太棒了！Python 是一门非常强大且友好的语言，无论是初学者还是资深开发者都对它爱不释手。它的语法简洁清晰，应用领域极其广泛，你选择了一个非常好的起点！👍

为了更好地帮助你，我们可以从几个方面来聊聊：

### 1. 你目前的学习阶段是？
*   **完全零基础**：正在学习变量、循环、函数这些基本概念？
*   **已入门**：掌握了基础语法，想做一些小项目练手？
*   **进阶中**：正在学习面向对象、模块、文件操作，或者某个特定库？

### 2. 你对Python的哪个应用方向最感兴趣？
这是决定你学习路径的关键。Python几乎无处不在：
*   **Web开发**：使用 `Django` 或 `Flask` 框架搭建网站和API。
*   **数据分析与可视化**：使用 `Pand

## 三种方式对比

| 方式 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 手动管理 | 简单、灵活 | 需要自己维护 | 简单应用 |
| 字典保存 | 结构化、可查询 | 代码稍复杂 | 需要日志记录 |
| 限制长度 | 节省内存 | 可能丢失早期对话 | 长对话 |

## 记忆的工作原理

```
用户输入
    ↓
加载记忆（从存储中读取历史）
    ↓
将历史 + 新输入传递给 LLM
    ↓
LLM 生成回答
    ↓
保存记忆（将新对话写入存储）
    ↓
输出回答
```

## 与其他模式的关系

| 第1章 提示链 | 第2章 路由 | 第3章 并行化 | 第4章 反思 | 第5章 工具使用 | 第6章 规划 | 第7章 多代理 | 第8章 记忆 |
|-------------|-----------|-------------|-----------|--------------|----------|-------------|----------|
| A → B → C | 选一条路 | A、B、C 同时执行 | 生成→批评→改进 | LLM + 外部工具 | 计划→执行 | 多个代理协作 | 保持对话状态 |
| 顺序执行 | 选择执行 | 并发执行 | 循环改进 | 扩展能力 | 结构化执行 | 分工合作 | 上下文保持 |

## 实际应用场景

- **聊天机器人**：记住用户偏好和对话历史
- **客服系统**：记住用户问题和解决方案
- **个人助手**：记住用户习惯和日程
- **教育系统**：记住学生学习进度